In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import csv, re, math
from pathlib import Path
import numpy as np

try:
    import rasterio as rio
except Exception:
    rio = None
    print("[WARN] rasterio not found; will try tifffile")
try:
    import tifffile as tiff
except Exception:
    tiff = None

REGION = "uttar_pradesh"  # <-- change to bangladesh / pak_punjab as needed

# Root to your final_data
ROOT = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data")

# Output base dir for .npy vectors + CSVs
OUT_BASE = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/aef_phase1")

# Which splits to prepare
SPLITS = ["train", "val", "test"]

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

# Regex to parse filenames like "25.2879_80.4283.ext"
LATLON_RE = re.compile(r"^(-?\d+(?:\.\d+)?)_(-?\d+(?:\.\d+)?)(?:\.[^.]+)?$")

def parse_lat_lon(stem: str):
    m = LATLON_RE.match(stem)
    if not m:
        return None
    lat = float(m.group(1))
    lon = float(m.group(2))
    return lat, lon

def read_aef_tif_to_vec(tif_path: Path) -> np.ndarray:
    arr = None
    if rio is not None:
        try:
            with rio.open(tif_path) as ds:
                arr = ds.read().astype(np.float32)  # [C,H,W]
        except Exception:
            arr = None
    if arr is None and tiff is not None:
        arr = tiff.imread(str(tif_path)).astype(np.float32)
        if arr.ndim == 3 and arr.shape[0] != 64 and arr.shape[-1] == 64:
            arr = np.moveaxis(arr, -1, 0)
    if arr is None or arr.ndim != 3:
        raise RuntimeError(f"Bad AEF TIF: {tif_path}")
    C = arr.shape[0]
    if C != 64:
        print(f"[WARN] Expected 64 bands, got {C} for {tif_path.name}")
    vec = arr.reshape(C, -1).mean(axis=1).astype(np.float32)  # [64]
    return vec

def build_stem_map(paths):
    """Return dict: stem -> full path (first wins)."""
    d = {}
    for p in paths:
        d.setdefault(p.stem, p)
    return d

def nearest_by_latlon(target_latlon, candidates_latlon, tol_deg=5e-4):
    """Return index of nearest candidate within tol_deg (degrees). Else None."""
    if not candidates_latlon:
        return None
    lat, lon = target_latlon
    best_i, best_d = None, 1e9
    for i, (clat, clon) in enumerate(candidates_latlon):
        d = math.hypot(lat - clat, lon - clon)
        if d < best_d:
            best_d, best_i = d, i
    return best_i if best_d <= tol_deg else None

def main():
    region_dir = ROOT / REGION
    assert region_dir.exists(), f"Missing region dir: {region_dir}"
    OUT_BASE.mkdir(parents=True, exist_ok=True)

    for split in SPLITS:
        img_dir = region_dir / split / "images"
        emb_dir = region_dir / split / "embeddings"
        if not img_dir.is_dir():
            raise RuntimeError(f"Expected folder not found: {img_dir}")
        if not emb_dir.is_dir():
            raise RuntimeError(f"Expected folder not found: {emb_dir}")

        out_vec_dir = OUT_BASE / REGION / split / "aef_vecs"
        out_vec_dir.mkdir(parents=True, exist_ok=True)

        # Collect files
        imgs = sorted([p for p in img_dir.iterglob("*") if p.suffix.lower() in IMG_EXTS])
        embs = sorted([p for p in emb_dir.iterglob("*.tif")])

        # Fast path: stem->path maps
        img_stem_map = build_stem_map(imgs)
        emb_stem_map = build_stem_map(embs)

        # Also build lat/lon lists for nearest matching when stems differ
        img_latlons = {p.stem: parse_lat_lon(p.stem) for p in imgs}
        emb_latlons = [parse_lat_lon(p.stem) for p in embs]

        csv_path = OUT_BASE / f"{REGION}_{split}_per_image_aef.csv"
        rows = []
        matched, fallback_matched = 0, 0

        for img in imgs:
            stem = img.stem

            # 1) Try exact stem match first (e.g., "25.2879_80.4283.*")
            emb_path = emb_stem_map.get(stem)

            # 2) If no exact match, try nearest by parsed lat/lon
            if emb_path is None:
                latlon = img_latlons.get(stem)
                if latlon is not None and all(x is not None for x in latlon):
                    idx = nearest_by_latlon(latlon, emb_latlons, tol_deg=5e-4)
                    if idx is not None:
                        emb_path = embs[idx]
                        fallback_matched += 1

            if emb_path is None:
                print(f"[MISS] No embedding GeoTIFF found for image {img.name}")
                continue

            # Convert TIF -> vector .npy (cached)
            out_npy = out_vec_dir / f"{stem}.npy"
            if not out_npy.exists():
                try:
                    vec = read_aef_tif_to_vec(emb_path)
                    np.save(out_npy, vec)
                except Exception as e:
                    print(f"[ERR] {emb_path.name}: {e}")
                    continue

            rows.append([REGION, img.name, str(out_npy)])
            matched += 1

        # write CSV
        with open(csv_path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["region", "filename", "aef_npy"])
            w.writerows(rows)

        print(f"[{REGION} / {split}] Images: {len(imgs)} | matched: {matched} "
              f"(exact:{matched - fallback_matched}, nearest:{fallback_matched})")
        print(f"[OK] Wrote CSV -> {csv_path}")

if __name__ == "__main__":
    main()

AttributeError: 'PosixPath' object has no attribute 'iterglob'